# Cranial Nerve Lesions — 9 Positions of Gaze

Simulates the standard 9-position clinical gaze test under cranial nerve lesions.
Each panel shows left (blue) and right (red) final eye positions after a saccade
to each target.  Misalignment between the two eyes reveals the palsy pattern.

| Lesion | What's zeroed | Expected finding |
|---|---|---|
| CN VI nerve (R) | `g_nerve[LR_R]` | R eye can't abduct; L eye normal |
| CN VI nucleus (R) | `g_nucleus[ABN_R]` | R eye can't abduct; L eye normal (no MLF in model) |
| CN III nerve (R) | `g_nerve[[MR_R,SR_R,IR_R,IO_R]]` | R eye fixed lateral+down; can't adduct/elevate |
| CN IV nerve (R) | `g_nerve[SO_R]` | Hypertropia in adduction; torsion deficit |
| Left INO | `g_mlf_ver_L = 0` | L eye adduction lag on rightward gaze (version drive to MR_L cut) |
| Right INO | `g_mlf_ver_R = 0` | R eye adduction lag on leftward gaze (version drive to MR_R cut) |
| Bilateral INO (BIMLF) | both = 0 | Both eyes can't adduct; exotropia on lateral gaze |

**INO modelling note:** The MLF pathway (ABN → contralateral MR) is absorbed into the
version_yaw component of CN3_MR in M_NUCLEUS.  `g_mlf_ver_L/R` scales that component;
vergence drive (vrg_yaw = +½) is preserved so convergence remains intact.


In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

from oculomotor.sim.simulator import PARAMS_DEFAULT, simulate, with_brain
from oculomotor.models.plant_models.muscle_geometry import (
    ABN_R,
    LR_R, MR_R, SR_R, IR_R, SO_R, IO_R,
    G_NUCLEUS_DEFAULT, G_NERVE_DEFAULT,
)

In [ ]:
# ── 9 gaze targets ─────────────────────────────────────────────────────────────
H = 20.0   # horizontal amplitude (deg)
V = 15.0   # vertical  amplitude (deg)

def _t(yaw_deg, pitch_deg):
    y, p = np.radians(yaw_deg), np.radians(pitch_deg)
    return np.array([np.sin(y)*np.cos(p), np.sin(p), np.cos(y)*np.cos(p)], dtype=np.float32)

TARGETS = {
    'Center'    : _t(  0,   0),
    'Right'     : _t(  H,   0),
    'Left'      : _t( -H,   0),
    'Up'        : _t(  0,   V),
    'Down'      : _t(  0,  -V),
    'Up-Right'  : _t(  H,   V),
    'Up-Left'   : _t( -H,   V),
    'Down-Right': _t(  H,  -V),
    'Down-Left' : _t( -H,  -V),
}

PT_CENTER = jnp.array([0., 0., 1.], dtype=jnp.float32)

In [ ]:
# ── Simulation helpers ─────────────────────────────────────────────────────────
# Target jumps from center at t=0 so the saccade happens in the plotted window.
# Warmup holds t[0] = center, settling the eyes at straight-ahead fixation.

T_SIM = 0.5
N     = int(T_SIM / 0.001) + 1
t_arr = jnp.linspace(0.0, T_SIM, N)
hv0   = jnp.zeros(N)

def _pt_jump(target_xyz):
    """Target array: center at t=0 (warmup), jumps to target for t>0."""
    return jnp.concatenate([
        PT_CENTER[None, :],
        jnp.tile(jnp.array(target_xyz), (N - 1, 1)),
    ])

def simulate_9(params, seed=42):
    """Run all 9 targets; return dict name → (L_pos, R_pos) final yaw/pitch (deg)."""
    results = {}
    for i, (name, pt) in enumerate(TARGETS.items()):
        eye = simulate(params, t_arr, head_vel_array=hv0,
                       p_target_array=_pt_jump(pt),
                       key=jax.random.PRNGKey(seed + i))
        results[name] = (
            np.array(eye[-1, :2]),   # left  eye (yaw, pitch)
            np.array(eye[-1, 3:5]),  # right eye (yaw, pitch)
        )
    return results

In [ ]:
# ── Lesion parameter sets ──────────────────────────────────────────────────────

# CN VI nerve (R) — LR_R paretic; nucleus and MR unaffected
params_cn6_nerve = with_brain(PARAMS_DEFAULT,
    g_nerve=G_NERVE_DEFAULT.at[LR_R].set(0.0))

# CN VI nucleus (R) — ABN_R: only ipsilateral LR_R affected (no MLF projection)
params_cn6_nuc = with_brain(PARAMS_DEFAULT,
    g_nucleus=G_NUCLEUS_DEFAULT.at[ABN_R].set(0.0))

# CN III nerve (R) — MR/SR/IR/IO paretic; LR and SO (CN IV) intact
_cn3_r_idx = jnp.array([MR_R, SR_R, IR_R, IO_R])
params_cn3 = with_brain(PARAMS_DEFAULT,
    g_nerve=G_NERVE_DEFAULT.at[_cn3_r_idx].set(0.0))

# CN IV nerve (R) — SO_R paretic
params_cn4 = with_brain(PARAMS_DEFAULT,
    g_nerve=G_NERVE_DEFAULT.at[SO_R].set(0.0))

# Left INO — version_yaw drive to left MR (CN3_MR_L) cut
#   On rightward gaze: R eye abducts normally (ABN_R intact), L eye can't adduct
#   Vergence (vrg_yaw component of CN3_MR_L) preserved
params_ino_L = with_brain(PARAMS_DEFAULT, g_mlf_ver_L=0.0)

# Right INO — version_yaw drive to right MR (CN3_MR_R) cut
#   On leftward gaze: L eye abducts normally, R eye can't adduct
params_ino_R = with_brain(PARAMS_DEFAULT, g_mlf_ver_R=0.0)

# Bilateral INO (BIMLF) — both MR subnuclei lose version drive
params_bimlf = with_brain(PARAMS_DEFAULT, g_mlf_ver_L=0.0, g_mlf_ver_R=0.0)

CONDITIONS = [
    ('Healthy',                          PARAMS_DEFAULT),
    ('CN VI nerve (R)\nLR_R = 0',        params_cn6_nerve),
    ('CN VI nucleus (R)\nABN_R = 0',     params_cn6_nuc),
    ('CN III nerve (R)\nMR/SR/IR/IO = 0',params_cn3),
    ('CN IV nerve (R)\nSO_R = 0',        params_cn4),
    ('Left INO\ng_mlf_ver_L = 0',        params_ino_L),
    ('Right INO\ng_mlf_ver_R = 0',       params_ino_R),
    ('Bilateral INO\nboth MLF ver = 0',  params_bimlf),
]

In [ ]:
# ── Run simulations ────────────────────────────────────────────────────────────
print("Running — first call compiles JIT (~1 min)...")
all_results = []
for title, params in CONDITIONS:
    print(f"  {title.split(chr(10))[0]}")
    all_results.append(simulate_9(params))
print("Done.")

In [ ]:
# ── 9-position plot ────────────────────────────────────────────────────────────
LABEL_OFFSETS = {
    'Center'    : (-2,  1.5), 'Right'     : ( 0.5,  1), 'Left'      : (-4,  1),
    'Up'        : (-2,  1.5), 'Down'      : (-2,  -2),
    'Up-Right'  : ( 0.5,  1), 'Up-Left'   : (-5,   1),
    'Down-Right': ( 0.5, -2), 'Down-Left' : (-5,  -2),
}

def plot_9(ax, results, title, show_labels=False):
    names = list(TARGETS.keys())
    L = np.array([results[n][0] for n in names])
    R = np.array([results[n][1] for n in names])
    for i in range(len(names)):
        ax.plot([L[i,0], R[i,0]], [L[i,1], R[i,1]], color='gray', lw=0.8, alpha=0.4, zorder=1)
    ax.scatter(L[:,0], L[:,1], c='royalblue', s=50, zorder=3, label='L eye')
    ax.scatter(R[:,0], R[:,1], c='crimson',   s=50, zorder=3, label='R eye', marker='s')
    if show_labels:
        for i, name in enumerate(names):
            dx, dy = LABEL_OFFSETS.get(name, (1, 1))
            ax.annotate(name, xy=L[i], xytext=(L[i,0]+dx, L[i,1]+dy), fontsize=6, color='royalblue')
    ax.axhline(0, color='k', lw=0.4, alpha=0.3)
    ax.axvline(0, color='k', lw=0.4, alpha=0.3)
    ax.set_xlabel('Yaw (deg)', fontsize=8)
    ax.set_ylabel('Pitch (deg)', fontsize=8)
    ax.set_title(title, fontsize=8)
    ax.set_xlim(-28, 28)
    ax.set_ylim(-22, 22)
    ax.set_aspect('equal')
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(True, alpha=0.15)

ncols = 4
nrows = 2
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 9))
axes_flat = axes.flatten()

for i, (ax, (title, _), results) in enumerate(zip(axes_flat, CONDITIONS, all_results)):
    plot_9(ax, results, title, show_labels=(i == 0))

fig.suptitle('9 Positions of Gaze  (blue circles = L eye,  red squares = R eye)',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/cn_lesions_9positions.png', dpi=130, bbox_inches='tight')
plt.show()

## Time series — saccade trajectories

INO is best revealed by the saccade trajectory (slow adducting eye), not the final position.  
The plots below compare yaw velocity for the adducting eye.

In [ ]:
def run_saccade(params, target_name, seed=99):
    pt = TARGETS[target_name]
    eye = simulate(params, t_arr, head_vel_array=hv0,
                   p_target_array=_pt_jump(pt),
                   key=jax.random.PRNGKey(seed))
    return np.array(eye)   # (N, 6)

t_np = np.array(t_arr)

fig2, axes2 = plt.subplots(3, 2, figsize=(13, 10))

# ── Row 0: CN VI nerve vs nucleus  (rightward saccade) ──────────────────────
cases_r = [
    ('Healthy',          PARAMS_DEFAULT,  'k'),
    ('CN VI nerve (R)',   params_cn6_nerve,'tab:blue'),
    ('CN VI nucleus (R)', params_cn6_nuc, 'tab:orange'),
]
for label, params, col in cases_r:
    eye = run_saccade(params, 'Right')
    axes2[0,0].plot(t_np, eye[:,0],  color=col, lw=1.3, ls='--', label=f'{label} L')
    axes2[0,0].plot(t_np, eye[:,3],  color=col, lw=1.3, ls='-',  label=f'{label} R')
    axes2[0,1].plot(t_np, np.gradient(eye[:,0], 0.001), color=col, lw=1.3, ls='--')
    axes2[0,1].plot(t_np, np.gradient(eye[:,3], 0.001), color=col, lw=1.3, ls='-')
axes2[0,0].set_title('Rightward saccade — position', fontsize=9)
axes2[0,1].set_title('Rightward saccade — velocity', fontsize=9)
axes2[0,0].legend(fontsize=7)

# ── Row 1: INO (left/right/bilateral)  ──────────────────────────────────────
# Left INO: rightward gaze, L eye adduction affected
cases_ino = [
    ('Healthy',        PARAMS_DEFAULT, 'k'),
    ('Left INO',       params_ino_L,   'tab:purple'),
    ('Right INO',      params_ino_R,   'tab:green'),
    ('Bilateral INO',  params_bimlf,   'tab:red'),
]
for label, params, col in cases_ino:
    eye = run_saccade(params, 'Right')
    axes2[1,0].plot(t_np, eye[:,0],  color=col, lw=1.3, ls='--', label=f'{label} L')
    axes2[1,0].plot(t_np, eye[:,3],  color=col, lw=1.3, ls='-',  label=f'{label} R')
    axes2[1,1].plot(t_np, np.gradient(eye[:,0], 0.001), color=col, lw=1.3, ls='--')
    axes2[1,1].plot(t_np, np.gradient(eye[:,3], 0.001), color=col, lw=1.3, ls='-')
axes2[1,0].set_title('Rightward saccade — position (INO)', fontsize=9)
axes2[1,1].set_title('Rightward saccade — velocity (INO)', fontsize=9)
axes2[1,0].legend(fontsize=7)

# ── Row 2: CN III / CN IV (leftward and down-left) ──────────────────────────
cases_v = [
    ('Healthy',         PARAMS_DEFAULT, 'k',          'Left'),
    ('CN III nerve (R)', params_cn3,    'tab:blue',    'Left'),
    ('CN IV nerve (R)',  params_cn4,    'tab:orange',  'Down-Left'),
]
for label, params, col, tgt in cases_v:
    eye = run_saccade(params, tgt)
    axes2[2,0].plot(t_np, eye[:,0],  color=col, lw=1.3, ls='--', label=f'{label} L (yaw)')
    axes2[2,0].plot(t_np, eye[:,3],  color=col, lw=1.3, ls='-',  label=f'{label} R (yaw)')
    axes2[2,1].plot(t_np, eye[:,1],  color=col, lw=1.3, ls='--', label=f'{label} L (pitch)')
    axes2[2,1].plot(t_np, eye[:,4],  color=col, lw=1.3, ls='-',  label=f'{label} R (pitch)')
axes2[2,0].set_title('Leftward saccade — yaw  (CN III: R eye adduction)', fontsize=9)
axes2[2,1].set_title('Down-Left saccade — pitch  (CN IV: R eye depression)', fontsize=9)
axes2[2,0].legend(fontsize=7)
axes2[2,1].legend(fontsize=7)

for row in axes2:
    for ax in row:
        ax.set_xlabel('Time (s)', fontsize=8)
        ax.grid(True, alpha=0.2)
        ax.annotate('-- L eye  /  solid R eye', xy=(0.02, 0.05), xycoords='axes fraction',
                    fontsize=6.5, color='gray')

axes2[0,0].set_ylabel('Eye yaw (deg)', fontsize=8)
axes2[1,0].set_ylabel('Eye yaw (deg)', fontsize=8)
axes2[0,1].set_ylabel('Yaw velocity (deg/s)', fontsize=8)
axes2[1,1].set_ylabel('Yaw velocity (deg/s)', fontsize=8)

fig2.suptitle('Saccade trajectories — CN lesions and INO', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/cn_lesions_timeseries.png', dpi=130, bbox_inches='tight')
plt.show()

## Graded CN VI nerve palsy and partial INO

Gains can be partial (0 < g < 1) for incomplete palsies or recovery.

In [ ]:
gains = [1.0, 0.75, 0.5, 0.25, 0.0]
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(gains)))
pt_right = _pt_jump(TARGETS['Right'])

fig3, axes3 = plt.subplots(1, 2, figsize=(12, 4))

for g, col in zip(gains, colors):
    # Graded CN VI nerve
    p_cn6 = with_brain(PARAMS_DEFAULT, g_nerve=G_NERVE_DEFAULT.at[LR_R].set(g))
    eye_cn6 = np.array(simulate(p_cn6, t_arr, head_vel_array=hv0,
                                p_target_array=pt_right, key=jax.random.PRNGKey(0)))
    axes3[0].plot(t_np, eye_cn6[:, 3], color=col, lw=1.5, label=f'g={g:.2f}')

    # Graded left INO — scale version_yaw component of CN3_MR_L
    p_ino = with_brain(PARAMS_DEFAULT, g_mlf_ver_L=g)
    eye_ino = np.array(simulate(p_ino, t_arr, head_vel_array=hv0,
                                p_target_array=pt_right, key=jax.random.PRNGKey(0)))
    axes3[1].plot(t_np, eye_ino[:, 0], color=col, lw=1.5, label=f'g={g:.2f}')

axes3[0].set_title('Graded CN VI nerve palsy (R)\nRight eye yaw — rightward saccade')
axes3[1].set_title('Graded left INO  (g_mlf_ver_L)\nLeft eye yaw — rightward saccade')
for ax in axes3:
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Eye yaw (deg)')
    ax.legend(title='gain', fontsize=8)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('../outputs/cn_lesions_graded.png', dpi=130, bbox_inches='tight')
plt.show()